<a href="https://colab.research.google.com/github/shreyasat27/mahework2025/blob/main/uhf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install block2
!pip install pyscf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.1 MB/s eta 0:00:00
  Attempting uninstall: tbb
    Found existing installation: tbb 2022.2.0
    Uninstalling tbb-2022.2.0:
      Successfully uninstalled tbb-2022.2.0
  Attempting uninstall: intel-openmp
    Found existing installation: intel-openmp 2025.2.1
    Uninstalling intel-openmp-2025.2.1:
      Successfully uninstalled intel-openmp-2025.2.1
  Attempting uninstall: mkl
    Found existing installation: mkl 2025.2.0
    Uninstalling mkl-2025.2.0:
      Successfully uninstalled mkl-2025.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:
import numpy as np
from pyblock2._pyscf.ao2mo import integrals as itg
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

bond_dims = [250] * 4 + [500] * 4
noises = [1e-4] * 4 + [1e-5] * 4 + [0]
thrds = [1e-10] * 8

RHF Multiplicity = 1,3

In [ ]:
from pyscf import gto, scf

mol = gto.M(
        atom="""
         C                 -1.47834026   -0.80121722    0.03464801
         H                 -1.03487388   -1.76406150    0.18014186
         H                 -2.51242343   -0.74719088   -0.23489447
         C                 -0.59814974    0.46185508   -0.00424973
         H                 -1.03606090    1.41246883   -0.22666220
         H                  0.45132235    0.38883051    0.19113638""", basis="sto3g", symmetry="Cs", verbose=0) #here spin =0
mf = scf.RHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_rhf_integrals(mf,
    ncore=0, ncas=None, g2e_symm=8)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=4)
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

pdm1 = driver.get_1pdm(ket)
pdm2 = driver.get_2pdm(ket).transpose(0, 3, 1, 2)
print('Energy from pdms = %20.15f' % (np.einsum('ij,ij->', pdm1, h1e)
    + 0.5 * np.einsum('ijkl,ijkl->', pdm2, driver.unpack_g2e(g2e)) + ecore))

impo = driver.get_identity_mpo()
expt = driver.expectation(ket, mpo, ket) / driver.expectation(ket, impo, ket)
print('Energy from expectation = %20.15f' % expt)

integral symmetrize error =  1.2096349154588091e-06
integral cutoff error =  0.0
mpo terms =      11295

Build MPO | Nsites =    14 | Nterms =      11295 | Algorithm = FastBIP | Cutoff = 1.00e-20
 Site =     0 /    14 .. Mmpo =    13 DW = 0.00e+00 NNZ =       13 SPT = 0.0000 Tmvc = 0.001 T = 0.006
 Site =     1 /    14 .. Mmpo =    42 DW = 0.00e+00 NNZ =      170 SPT = 0.6886 Tmvc = 0.001 T = 0.007
 Site =     2 /    14 .. Mmpo =    64 DW = 0.00e+00 NNZ =      349 SPT = 0.8702 Tmvc = 0.002 T = 0.007
 Site =     3 /    14 .. Mmpo =    94 DW = 0.00e+00 NNZ =      574 SPT = 0.9046 Tmvc = 0.002 T = 0.008
 Site =     4 /    14 .. Mmpo =   140 DW = 0.00e+00 NNZ =      599 SPT = 0.9545 Tmvc = 0.002 T = 0.009
 Site =     5 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =      865 SPT = 0.9668 Tmvc = 0.003 T = 0.013
 Site =     6 /    14 .. Mmpo =   240 DW = 0.00e+00 NNZ =     1265 SPT = 0.9717 Tmvc = 0.004 T = 0.017
 Site =     7 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =     5198 SPT = 0.8836 Tmv

In [ ]:
from pyscf import gto, scf

mol = gto.M(
        atom="""
         C                 -1.47834026   -0.80121722    0.03464801
         H                 -1.03487388   -1.76406150    0.18014186
         H                 -2.51242343   -0.74719088   -0.23489447
         C                 -0.59814974    0.46185508   -0.00424973
         H                 -1.03606090    1.41246883   -0.22666220
         H                  0.45132235    0.38883051    0.19113638""", basis="sto3g", spin =2, symmetry="Cs", verbose=0) #here spin =0
mf = scf.RHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_rhf_integrals(mf,
    ncore=0, ncas=None, g2e_symm=8)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=4)
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

pdm1 = driver.get_1pdm(ket)
pdm2 = driver.get_2pdm(ket).transpose(0, 3, 1, 2)
print('Energy from pdms = %20.15f' % (np.einsum('ij,ij->', pdm1, h1e)
    + 0.5 * np.einsum('ijkl,ijkl->', pdm2, driver.unpack_g2e(g2e)) + ecore))

impo = driver.get_identity_mpo()
expt = driver.expectation(ket, mpo, ket) / driver.expectation(ket, impo, ket)
print('Energy from expectation = %20.15f' % expt)

integral symmetrize error =  1.22735935275229e-06
integral cutoff error =  0.0
mpo terms =      11295

Build MPO | Nsites =    14 | Nterms =      11295 | Algorithm = FastBIP | Cutoff = 1.00e-20
 Site =     0 /    14 .. Mmpo =    13 DW = 0.00e+00 NNZ =       13 SPT = 0.0000 Tmvc = 0.002 T = 0.009
 Site =     1 /    14 .. Mmpo =    42 DW = 0.00e+00 NNZ =      170 SPT = 0.6886 Tmvc = 0.002 T = 0.010
 Site =     2 /    14 .. Mmpo =    64 DW = 0.00e+00 NNZ =      349 SPT = 0.8702 Tmvc = 0.002 T = 0.007
 Site =     3 /    14 .. Mmpo =    94 DW = 0.00e+00 NNZ =      573 SPT = 0.9048 Tmvc = 0.002 T = 0.008
 Site =     4 /    14 .. Mmpo =   140 DW = 0.00e+00 NNZ =      597 SPT = 0.9546 Tmvc = 0.002 T = 0.008
 Site =     5 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =      865 SPT = 0.9668 Tmvc = 0.002 T = 0.009
 Site =     6 /    14 .. Mmpo =   240 DW = 0.00e+00 NNZ =     1263 SPT = 0.9717 Tmvc = 0.002 T = 0.010
 Site =     7 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =     5187 SPT = 0.8838 Tmvc 

UHF for multiplicity 1,3

In [ ]:
from pyscf import gto, scf

mol = gto.M(
        atom="""
         C                 -1.47834026   -0.80121722    0.03464801
         H                 -1.03487388   -1.76406150    0.18014186
         H                 -2.51242343   -0.74719088   -0.23489447
         C                 -0.59814974    0.46185508   -0.00424973
         H                 -1.03606090    1.41246883   -0.22666220
         H                  0.45132235    0.38883051    0.19113638""", basis="sto3g", symmetry="Cs", verbose=0) #here spin =0
mf = scf.UHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_uhf_integrals(mf,
    ncore=0, ncas=None, g2e_symm=8)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SZ, n_threads=4)
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

integral symmetrize error =  3.885140396822741e-06
integral cutoff error =  0.0
mpo terms =      31934

Build MPO | Nsites =    14 | Nterms =      31934 | Algorithm = FastBIP | Cutoff = 1.00e-20
 Site =     0 /    14 .. Mmpo =    26 DW = 0.00e+00 NNZ =       26 SPT = 0.0000 Tmvc = 0.003 T = 0.011
 Site =     1 /    14 .. Mmpo =    82 DW = 0.00e+00 NNZ =      419 SPT = 0.8035 Tmvc = 0.005 T = 0.018
 Site =     2 /    14 .. Mmpo =   126 DW = 0.00e+00 NNZ =      891 SPT = 0.9138 Tmvc = 0.005 T = 0.019
 Site =     3 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =     1507 SPT = 0.9357 Tmvc = 0.005 T = 0.018
 Site =     4 /    14 .. Mmpo =   278 DW = 0.00e+00 NNZ =     1555 SPT = 0.9699 Tmvc = 0.006 T = 0.020
 Site =     5 /    14 .. Mmpo =   370 DW = 0.00e+00 NNZ =     2303 SPT = 0.9776 Tmvc = 0.006 T = 0.025
 Site =     6 /    14 .. Mmpo =   478 DW = 0.00e+00 NNZ =     3371 SPT = 0.9809 Tmvc = 0.006 T = 0.023
 Site =     7 /    14 .. Mmpo =   370 DW = 0.00e+00 NNZ =    14547 SPT = 0.9177 Tmvc

In [ ]:
from pyscf import gto, scf
import numpy as np
from pyblock2._pyscf.ao2mo import integrals as itg
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

mol = gto.M(
        atom="""
         C                 -1.47834026   -0.80121722    0.03464801
         H                 -1.03487388   -1.76406150    0.18014186
         H                 -2.51242343   -0.74719088   -0.23489447
         C                 -0.59814974    0.46185508   -0.00424973
         H                 -1.03606090    1.41246883   -0.22666220
         H                  0.45132235    0.38883051    0.19113638""", basis="sto3g", spin=2, symmetry="Cs", verbose=0)
mf = scf.UHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_uhf_integrals(mf, ncore=0, ncas=None, g2e_symm=8)

# FIX: Handle UHF tuples
if isinstance(h1e, tuple): h1e = h1e[0]
if isinstance(g2e, tuple) and len(g2e) > 0: g2e = g2e[0]
if isinstance(orb_sym, tuple): orb_sym = orb_sym[0]
orb_sym = np.array(orb_sym, dtype=np.int32)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=4)  # SU2 for spin
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

integral symmetrize error =  1.2251548446555034e-06
integral cutoff error =  0.0
mpo terms =      11307

Build MPO | Nsites =    14 | Nterms =      11307 | Algorithm = FastBIP | Cutoff = 1.00e-20
 Site =     0 /    14 .. Mmpo =    13 DW = 0.00e+00 NNZ =       13 SPT = 0.0000 Tmvc = 0.002 T = 0.007
 Site =     1 /    14 .. Mmpo =    42 DW = 0.00e+00 NNZ =      170 SPT = 0.6886 Tmvc = 0.003 T = 0.010
 Site =     2 /    14 .. Mmpo =    64 DW = 0.00e+00 NNZ =      347 SPT = 0.8709 Tmvc = 0.002 T = 0.009
 Site =     3 /    14 .. Mmpo =    94 DW = 0.00e+00 NNZ =      572 SPT = 0.9049 Tmvc = 0.002 T = 0.010
 Site =     4 /    14 .. Mmpo =   140 DW = 0.00e+00 NNZ =      598 SPT = 0.9546 Tmvc = 0.003 T = 0.011
 Site =     5 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =     1055 SPT = 0.9595 Tmvc = 0.003 T = 0.012
 Site =     6 /    14 .. Mmpo =   240 DW = 0.00e+00 NNZ =     1075 SPT = 0.9759 Tmvc = 0.003 T = 0.012
 Site =     7 /    14 .. Mmpo =   186 DW = 0.00e+00 NNZ =     5184 SPT = 0.8839 Tmv

UHF for Para. Benzene

In [ ]:
from pyscf import gto, scf

mol = gto.M(
        atom="""
          C                 -0.39103273    0.09258996   -0.07178749
          C                  1.00623741   -0.01491512   -0.07266507
          C                  1.79797767    1.14137238   -0.06438697
          C                  1.19244779    2.40516496   -0.05523027
          C                 -0.20482235    2.51267005   -0.05435395
          C                 -0.99656261    1.35638255   -0.06263153
          H                 -0.99554399   -0.79026121   -0.07810794
          H                  1.46857291   -0.97984882   -0.07965397
          H                  1.79695906    3.28801613   -0.04890939
          H                 -0.66715785    3.47760376   -0.04736661
          C                  3.33343937    1.02323494   -0.06535247
          H                  3.86679411    0.98887456   -0.99231098
          H                  3.86693138    0.97551280    0.86093521
          C                 -2.53202430    1.47451999   -0.06166628
          H                 -3.06537916    1.50888086    0.86529215
          H                 -3.06551621    1.52224164   -0.98795405""", basis="sto3g", verbose=0) #here spin =0
mf = scf.UHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_uhf_integrals(mf,
    ncore=0, ncas=None, g2e_symm=8)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SZ, n_threads=4)
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

integral symmetrize error =  0.0
integral cutoff error =  0.0
mpo terms =    8081280

Build MPO | Nsites =    48 | Nterms =    8081280 | Algorithm = FastBIP | Cutoff = 1.00e-20
 Site =     0 /    48 .. Mmpo =    26 DW = 0.00e+00 NNZ =       26 SPT = 0.0000 Tmvc = 0.547 T = 1.669
 Site =     1 /    48 .. Mmpo =   106 DW = 0.00e+00 NNZ =      123 SPT = 0.9554 Tmvc = 0.461 T = 2.236
 Site =     2 /    48 .. Mmpo =   278 DW = 0.00e+00 NNZ =      367 SPT = 0.9875 Tmvc = 0.471 T = 1.634
 Site =     3 /    48 .. Mmpo =   338 DW = 0.00e+00 NNZ =    17483 SPT = 0.8139 Tmvc = 0.790 T = 2.362
 Site =     4 /    48 .. Mmpo =   414 DW = 0.00e+00 NNZ =    15499 SPT = 0.8892 Tmvc = 0.652 T = 2.783
 Site =     5 /    48 .. Mmpo =   506 DW = 0.00e+00 NNZ =    22695 SPT = 0.8917 Tmvc = 0.851 T = 2.703
 Site =     6 /    48 .. Mmpo =   614 DW = 0.00e+00 NNZ =    31043 SPT = 0.9001 Tmvc = 0.852 T = 2.918
 Site =     7 /    48 .. Mmpo =   738 DW = 0.00e+00 NNZ =    40435 SPT = 0.9108 Tmvc = 0.986 T = 3.417

In [ ]:
from pyscf import gto, scf
import numpy as np
from pyblock2._pyscf.ao2mo import integrals as itg
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

mol = gto.M(
        atom="""
          C                 -0.39103273    0.09258996   -0.07178749
          C                  1.00623741   -0.01491512   -0.07266507
          C                  1.79797767    1.14137238   -0.06438697
          C                  1.19244779    2.40516496   -0.05523027
          C                 -0.20482235    2.51267005   -0.05435395
          C                 -0.99656261    1.35638255   -0.06263153
          H                 -0.99554399   -0.79026121   -0.07810794
          H                  1.46857291   -0.97984882   -0.07965397
          H                  1.79695906    3.28801613   -0.04890939
          H                 -0.66715785    3.47760376   -0.04736661
          C                  3.33343937    1.02323494   -0.06535247
          H                  3.86679411    0.98887456   -0.99231098
          H                  3.86693138    0.97551280    0.86093521
          C                 -2.53202430    1.47451999   -0.06166628
          H                 -3.06537916    1.50888086    0.86529215
          H                 -3.06551621    1.52224164   -0.98795405""", basis="sto3g", spin=2, verbose=0)
mf = scf.UHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_uhf_integrals(mf, ncore=0, ncas=None, g2e_symm=8)

# FIX: Handle UHF tuples
if isinstance(h1e, tuple): h1e = h1e[0]
if isinstance(g2e, tuple) and len(g2e) > 0: g2e = g2e[0]
if isinstance(orb_sym, tuple): orb_sym = orb_sym[0]
orb_sym = np.array(orb_sym, dtype=np.int32)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=4)  # SU2 for spin
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

UHF for Meta Benzene

In [ ]:
from pyscf import gto, scf

mol = gto.M(
        atom="""
             C                  0.10738604    0.67464760   -0.12467558
             C                  1.49918579    0.69895458    0.03726450
             C                  2.18606599    1.92026062    0.01426229
             C                  1.48114652    3.11725963   -0.17068053
             C                  0.08934688    3.09295257   -0.33262158
             C                 -0.59753322    1.87164646   -0.30962024
             H                  2.03740717   -0.21498053    0.17847486
             H                  2.00559467    4.04975399   -0.18824210
             H                 -0.44887454    4.00688771   -0.47383161
             H                 -1.66020328    1.85308758   -0.43326461
             C                  3.71551620    1.94697163    0.19221901
             H                  4.35491139    1.87820541   -0.66296811
             H                  4.13879096    2.03429680    1.17105137
             C                 -0.64742769   -0.66744672   -0.09939582
             H                 -0.80159206   -1.21174110   -1.00762320
             H                 -1.01771181   -1.05564646    0.82639608""", basis="sto3g", verbose=0) #here spin =0
mf = scf.UHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_uhf_integrals(mf,ncore=0, ncas=None, g2e_symm=8)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SZ, n_threads=4)
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)

In [ ]:
from pyscf import gto, scf
import numpy as np
from pyblock2._pyscf.ao2mo import integrals as itg
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

mol = gto.M(
        atom="""
             C                  0.10738604    0.67464760   -0.12467558
             C                  1.49918579    0.69895458    0.03726450
             C                  2.18606599    1.92026062    0.01426229
             C                  1.48114652    3.11725963   -0.17068053
             C                  0.08934688    3.09295257   -0.33262158
             C                 -0.59753322    1.87164646   -0.30962024
             H                  2.03740717   -0.21498053    0.17847486
             H                  2.00559467    4.04975399   -0.18824210
             H                 -0.44887454    4.00688771   -0.47383161
             H                 -1.66020328    1.85308758   -0.43326461
             C                  3.71551620    1.94697163    0.19221901
             H                  4.35491139    1.87820541   -0.66296811
             H                  4.13879096    2.03429680    1.17105137
             C                 -0.64742769   -0.66744672   -0.09939582
             H                 -0.80159206   -1.21174110   -1.00762320
             H                 -1.01771181   -1.05564646    0.82639608""", basis="sto3g", spin=2, verbose=0)
mf = scf.UHF(mol).run(conv_tol=1E-14)
ncas, n_elec, spin, ecore, h1e, g2e, orb_sym = itg.get_uhf_integrals(mf, ncore=0, ncas=None, g2e_symm=8)

# FIX: Handle UHF tuples
if isinstance(h1e, tuple): h1e = h1e[0]
if isinstance(g2e, tuple) and len(g2e) > 0: g2e = g2e[0]
if isinstance(orb_sym, tuple): orb_sym = orb_sym[0]
orb_sym = np.array(orb_sym, dtype=np.int32)

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=4)  # SU2 for spin
driver.initialize_system(n_sites=ncas, n_elec=n_elec, spin=spin, orb_sym=orb_sym)

mpo = driver.get_qc_mpo(h1e=h1e, g2e=g2e, ecore=ecore, iprint=1)
ket = driver.get_random_mps(tag="GS", bond_dim=250, nroots=1)
energy = driver.dmrg(mpo, ket, n_sweeps=20, bond_dims=bond_dims, noises=noises,
    thrds=thrds, iprint=1)
print('DMRG energy = %20.15f' % energy)